# Etapa 1 - Leitura da base e Análise de Dados

## Inicializacao do ambiente Spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PySpark Transactions Classification")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

## Leitura da base de dados

In [2]:
# A leitura via CSV foi usada apenas na etapa inicial de conversao;
# depois disso, mantemos a leitura em Parquet por melhor performance no Spark.

# dataframe = (
#     spark.read
#     .csv("./synthetic_fraud_data.csv", header=True, inferSchema=True)
# )
#
# dataframe.write.parquet("synthetic_fraud_data.parquet")

dataframe = (
    spark.read
    .parquet("./synthetic_fraud_data.parquet")
)

dataframe.createOrReplaceTempView("transactions_raw")

rows_count = dataframe.count()
print(f"Número de linhas no DataFrame: {rows_count}")

Número de linhas no DataFrame: 7483766


## Estrutura e visualização inicial da base

In [3]:
dataframe.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- merchant_type: string (nullable = true)
 |-- merchant: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- city_size: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_present: boolean (nullable = true)
 |-- device: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- device_fingerprint: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- distance_from_home: integer (nullable = true)
 |-- high_risk_merchant: boolean (nullable = true)
 |-- transaction_hour: integer (nullable = true)
 |-- weekend_transaction: boolean (nullable = true)
 |-- velocity_last_hour: string (nu

In [4]:
dataframe.show()

+--------------+-----------+----------------+--------------------+-----------------+-------------+--------------------+---------+--------+---------+------------+---------+---------------+------------+-----------+-------+--------------------+---------------+------------------+------------------+----------------+-------------------+--------------------+--------+
|transaction_id|customer_id|     card_number|           timestamp|merchant_category|merchant_type|            merchant|   amount|currency|  country|        city|city_size|      card_type|card_present|     device|channel|  device_fingerprint|     ip_address|distance_from_home|high_risk_merchant|transaction_hour|weekend_transaction|  velocity_last_hour|is_fraud|
+--------------+-----------+----------------+--------------------+-----------------+-------------+--------------------+---------+--------+---------+------------+---------+---------------+------------+-----------+-------+--------------------+---------------+-----------------

## Analise inicial da coluna alvo, nulos e duplicatas

In [5]:
spark.sql("SELECT is_fraud ,COUNT(is_fraud) FROM transactions_raw GROUP BY is_fraud ORDER BY is_fraud").show()

+--------+---------------+
|is_fraud|count(is_fraud)|
+--------+---------------+
|   false|        5989047|
|    true|        1494719|
+--------+---------------+



In [6]:
spark.sql("""
    SELECT
        SUM(CASE WHEN transaction_id IS NULL THEN 1 ELSE 0 END) AS transaction_id_nulls,
        SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS customer_id_nulls,
        SUM(CASE WHEN card_number IS NULL THEN 1 ELSE 0 END) AS card_number_nulls,
        SUM(CASE WHEN timestamp IS NULL THEN 1 ELSE 0 END) AS timestamp_nulls,
        SUM(CASE WHEN merchant_category IS NULL THEN 1 ELSE 0 END) AS merchant_category_nulls,
        SUM(CASE WHEN merchant_type IS NULL THEN 1 ELSE 0 END) AS merchant_type_nulls,
        SUM(CASE WHEN merchant IS NULL THEN 1 ELSE 0 END) AS merchant_nulls,
        SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END) AS amount_nulls,
        SUM(CASE WHEN currency IS NULL THEN 1 ELSE 0 END) AS currency_nulls,
        SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END) AS country_nulls,
        SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) AS city_nulls,
        SUM(CASE WHEN city_size IS NULL THEN 1 ELSE 0 END) AS city_size_nulls,
        SUM(CASE WHEN card_type IS NULL THEN 1 ELSE 0 END) AS card_type_nulls,
        SUM(CASE WHEN card_present IS NULL THEN 1 ELSE 0 END) AS card_present_nulls,
        SUM(CASE WHEN device IS NULL THEN 1 ELSE 0 END) AS device_nulls,
        SUM(CASE WHEN channel IS NULL THEN 1 ELSE 0 END) AS channel_nulls,
        SUM(CASE WHEN device_fingerprint IS NULL THEN 1 ELSE 0 END) AS device_fingerprint_nulls,
        SUM(CASE WHEN ip_address IS NULL THEN 1 ELSE 0 END) AS ip_address_nulls,
        SUM(CASE WHEN distance_from_home IS NULL THEN 1 ELSE 0 END) AS distance_from_home_nulls,
        SUM(CASE WHEN high_risk_merchant IS NULL THEN 1 ELSE 0 END) AS high_risk_merchant_nulls,
        SUM(CASE WHEN transaction_hour IS NULL THEN 1 ELSE 0 END) AS transaction_hour_nulls,
        SUM(CASE WHEN weekend_transaction IS NULL THEN 1 ELSE 0 END) AS weekend_transaction_nulls,
        SUM(CASE WHEN velocity_last_hour IS NULL THEN 1 ELSE 0 END) AS velocity_last_hour_nulls,
        SUM(CASE WHEN is_fraud IS NULL THEN 1 ELSE 0 END) AS is_fraud_nulls
    FROM transactions_raw
""").show(truncate=False, vertical=True)

-RECORD 0------------------------
 transaction_id_nulls      | 0   
 customer_id_nulls         | 0   
 card_number_nulls         | 0   
 timestamp_nulls           | 0   
 merchant_category_nulls   | 0   
 merchant_type_nulls       | 0   
 merchant_nulls            | 0   
 amount_nulls              | 0   
 currency_nulls            | 0   
 country_nulls             | 0   
 city_nulls                | 0   
 city_size_nulls           | 0   
 card_type_nulls           | 0   
 card_present_nulls        | 0   
 device_nulls              | 0   
 channel_nulls             | 0   
 device_fingerprint_nulls  | 0   
 ip_address_nulls          | 0   
 distance_from_home_nulls  | 0   
 high_risk_merchant_nulls  | 0   
 transaction_hour_nulls    | 0   
 weekend_transaction_nulls | 0   
 velocity_last_hour_nulls  | 0   
 is_fraud_nulls            | 0   



In [7]:
spark.sql("""
    SELECT COUNT(*) AS duplicated_rows
    FROM (
        SELECT
            transaction_id, customer_id, card_number, timestamp, merchant_category,
            merchant_type, merchant, amount, currency, country, city, city_size,
            card_type, card_present, device, channel, device_fingerprint, ip_address,
            distance_from_home, high_risk_merchant, transaction_hour,
            weekend_transaction, velocity_last_hour, is_fraud,
            COUNT(*) AS qty
        FROM transactions_raw
        GROUP BY
            transaction_id, customer_id, card_number, timestamp, merchant_category,
            merchant_type, merchant, amount, currency, country, city, city_size,
            card_type, card_present, device, channel, device_fingerprint, ip_address,
            distance_from_home, high_risk_merchant, transaction_hour,
            weekend_transaction, velocity_last_hour, is_fraud
        HAVING COUNT(*) > 1
    ) t
""").show()

+---------------+
|duplicated_rows|
+---------------+
|              0|
+---------------+



## Limpeza inicial

In [8]:
spark.sql("""
    SELECT
        SUM(CASE WHEN TRIM(transaction_id) = '' THEN 1 ELSE 0 END) AS transaction_id_empty,
        SUM(CASE WHEN TRIM(customer_id) = '' THEN 1 ELSE 0 END) AS customer_id_empty,
        SUM(CASE WHEN TRIM(merchant_category) = '' THEN 1 ELSE 0 END) AS merchant_category_empty,
        SUM(CASE WHEN TRIM(merchant_type) = '' THEN 1 ELSE 0 END) AS merchant_type_empty,
        SUM(CASE WHEN TRIM(merchant) = '' THEN 1 ELSE 0 END) AS merchant_empty,
        SUM(CASE WHEN TRIM(currency) = '' THEN 1 ELSE 0 END) AS currency_empty,
        SUM(CASE WHEN TRIM(country) = '' THEN 1 ELSE 0 END) AS country_empty,
        SUM(CASE WHEN TRIM(city) = '' THEN 1 ELSE 0 END) AS city_empty,
        SUM(CASE WHEN TRIM(city_size) = '' THEN 1 ELSE 0 END) AS city_size_empty,
        SUM(CASE WHEN TRIM(card_type) = '' THEN 1 ELSE 0 END) AS card_type_empty,
        SUM(CASE WHEN TRIM(device) = '' THEN 1 ELSE 0 END) AS device_empty,
        SUM(CASE WHEN TRIM(channel) = '' THEN 1 ELSE 0 END) AS channel_empty,
        SUM(CASE WHEN TRIM(device_fingerprint) = '' THEN 1 ELSE 0 END) AS device_fingerprint_empty,
        SUM(CASE WHEN TRIM(ip_address) = '' THEN 1 ELSE 0 END) AS ip_address_empty,
        SUM(CASE WHEN TRIM(velocity_last_hour) = '' THEN 1 ELSE 0 END) AS velocity_last_hour_empty
    FROM transactions_raw
""").show(truncate=False, vertical=True)

-RECORD 0-----------------------
 transaction_id_empty     | 0   
 customer_id_empty        | 0   
 merchant_category_empty  | 0   
 merchant_type_empty      | 0   
 merchant_empty           | 0   
 currency_empty           | 0   
 country_empty            | 0   
 city_empty               | 0   
 city_size_empty          | 0   
 card_type_empty          | 0   
 device_empty             | 0   
 channel_empty            | 0   
 device_fingerprint_empty | 0   
 ip_address_empty         | 0   
 velocity_last_hour_empty | 0   



# Etapa 2 - Pre-processamento e selecao inicial de atributos

## Selecao inicial de features

In [9]:
selected_dataframe = spark.sql("""
    SELECT
        merchant_category,
        merchant_type,
        amount,
        currency,
        country,
        city,
        city_size,
        card_type,
        card_present,
        device,
        channel,
        distance_from_home,
        high_risk_merchant,
        transaction_hour,
        weekend_transaction,
        velocity_last_hour,
        is_fraud
    FROM transactions_raw
""")

selected_dataframe.createOrReplaceTempView("transactions_selected")

In [10]:
spark.sql("SELECT * FROM transactions_selected LIMIT 10").show(truncate=False)

+-----------------+-------------+---------+--------+---------+------------+---------+---------------+------------+-----------+-------+------------------+------------------+----------------+-------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+--------+
|merchant_category|merchant_type|amount   |currency|country  |city        |city_size|card_type      |card_present|device     |channel|distance_from_home|high_risk_merchant|transaction_hour|weekend_transaction|velocity_last_hour                                                                                                                                    |is_fraud|
+-----------------+-------------+---------+--------+---------+------------+---------+---------------+------------+-----------+-------+------------------+------------------+----------------+-------------------+---------------------------------------------------

## Inspecao inicial de velocity_last_hour

In [11]:
spark.sql("SELECT velocity_last_hour FROM transactions_selected LIMIT 10").show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------+
|velocity_last_hour                                                                                                                                    |
+------------------------------------------------------------------------------------------------------------------------------------------------------+
|{'num_transactions': 222, 'total_amount': 6001328.808632168, 'unique_merchants': 87, 'unique_countries': 12, 'max_single_amount': 1108920.8766736432} |
|{'num_transactions': 110, 'total_amount': 6437171.840036886, 'unique_merchants': 70, 'unique_countries': 9, 'max_single_amount': 2574176.473608244}   |
|{'num_transactions': 268, 'total_amount': 12635383.309312332, 'unique_merchants': 94, 'unique_countries': 12, 'max_single_amount': 3419844.28627481}  |
|{'num_transactions': 21, 'total_amount': 223751.77196771436, 'unique_merchants': 

In [12]:
velocity_preview_dataframe = spark.sql("""
    SELECT
        velocity_last_hour,
        velocity_struct.num_transactions AS num_transactions_last_hour,
        velocity_struct.total_amount AS total_amount_last_hour,
        velocity_struct.unique_merchants AS unique_merchants_last_hour,
        velocity_struct.unique_countries AS unique_countries_last_hour,
        velocity_struct.max_single_amount AS max_single_amount_last_hour
    FROM (
        SELECT
            velocity_last_hour,
            from_json(
                replace(velocity_last_hour, '''', '"'),
                'num_transactions INT, total_amount DOUBLE, unique_merchants INT, unique_countries INT, max_single_amount DOUBLE'
            ) AS velocity_struct
        FROM transactions_selected
    ) t
    LIMIT 10
""")

velocity_preview_dataframe.show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+----------------------+--------------------------+--------------------------+---------------------------+
|velocity_last_hour                                                                                                                                    |num_transactions_last_hour|total_amount_last_hour|unique_merchants_last_hour|unique_countries_last_hour|max_single_amount_last_hour|
+------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+----------------------+--------------------------+--------------------------+---------------------------+
|{'num_transactions': 222, 'total_amount': 6001328.808632168, 'unique_merchants': 87, 'unique_countries': 12, 'max_single_amount': 1108920.876673

## Criacao da view tratada para a modelagem

In [13]:
treated_dataframe = spark.sql("""
    SELECT
        merchant_category,
        merchant_type,
        amount,
        currency,
        country,
        city,
        city_size,
        card_type,
        card_present,
        device,
        channel,
        distance_from_home,
        high_risk_merchant,
        transaction_hour,
        weekend_transaction,
        velocity_struct.num_transactions AS num_transactions_last_hour,
        velocity_struct.total_amount AS total_amount_last_hour,
        velocity_struct.unique_merchants AS unique_merchants_last_hour,
        velocity_struct.unique_countries AS unique_countries_last_hour,
        velocity_struct.max_single_amount AS max_single_amount_last_hour,
        is_fraud
    FROM (
        SELECT
            merchant_category,
            merchant_type,
            amount,
            currency,
            country,
            city,
            city_size,
            card_type,
            card_present,
            device,
            channel,
            distance_from_home,
            high_risk_merchant,
            transaction_hour,
            weekend_transaction,
            from_json(
                replace(velocity_last_hour, '''', '"'),
                'num_transactions INT, total_amount DOUBLE, unique_merchants INT, unique_countries INT, max_single_amount DOUBLE'
            ) AS velocity_struct,
            is_fraud
        FROM transactions_selected
    ) t
""")

treated_dataframe.createOrReplaceTempView("transactions_treated")

In [14]:
spark.sql("DESCRIBE transactions_treated").show(truncate=False)
spark.sql("SELECT * FROM transactions_treated LIMIT 10").show(truncate=False)

+---------------------------+---------+-------+
|col_name                   |data_type|comment|
+---------------------------+---------+-------+
|merchant_category          |string   |NULL   |
|merchant_type              |string   |NULL   |
|amount                     |double   |NULL   |
|currency                   |string   |NULL   |
|country                    |string   |NULL   |
|city                       |string   |NULL   |
|city_size                  |string   |NULL   |
|card_type                  |string   |NULL   |
|card_present               |boolean  |NULL   |
|device                     |string   |NULL   |
|channel                    |string   |NULL   |
|distance_from_home         |int      |NULL   |
|high_risk_merchant         |boolean  |NULL   |
|transaction_hour           |int      |NULL   |
|weekend_transaction        |boolean  |NULL   |
|num_transactions_last_hour |int      |NULL   |
|total_amount_last_hour     |double   |NULL   |
|unique_merchants_last_hour |int      |N

## Validacao do tratamento de velocity_last_hour

In [15]:
spark.sql("""
    SELECT
        SUM(CASE WHEN num_transactions_last_hour IS NULL THEN 1 ELSE 0 END) AS num_transactions_last_hour_nulls,
        SUM(CASE WHEN total_amount_last_hour IS NULL THEN 1 ELSE 0 END) AS total_amount_last_hour_nulls,
        SUM(CASE WHEN unique_merchants_last_hour IS NULL THEN 1 ELSE 0 END) AS unique_merchants_last_hour_nulls,
        SUM(CASE WHEN unique_countries_last_hour IS NULL THEN 1 ELSE 0 END) AS unique_countries_last_hour_nulls,
        SUM(CASE WHEN max_single_amount_last_hour IS NULL THEN 1 ELSE 0 END) AS max_single_amount_last_hour_nulls
    FROM transactions_treated
""").show(truncate=False, vertical=True)

-RECORD 0--------------------------------
 num_transactions_last_hour_nulls  | 0   
 total_amount_last_hour_nulls      | 0   
 unique_merchants_last_hour_nulls  | 0   
 unique_countries_last_hour_nulls  | 0   
 max_single_amount_last_hour_nulls | 0   



# Etapa 3 - Analise exploratoria de dados

## Distribuicao da classe alvo

In [16]:
spark.sql("""
    SELECT
        is_fraud,
        total_transactions,
        ROUND(100.0 * total_transactions / SUM(total_transactions) OVER (), 2) AS percentage
    FROM (
        SELECT
            is_fraud,
            COUNT(*) AS total_transactions
        FROM transactions_treated
        GROUP BY is_fraud
    ) t
    ORDER BY is_fraud
""").show(truncate=False)

+--------+------------------+----------+
|is_fraud|total_transactions|percentage|
+--------+------------------+----------+
|false   |5989047           |80.03     |
|true    |1494719           |19.97     |
+--------+------------------+----------+



## Estatisticas das variaveis numericas por classe

In [17]:
spark.sql("""
    SELECT
        is_fraud,
        ROUND(AVG(amount), 2) AS avg_amount,
        ROUND(MIN(amount), 2) AS min_amount,
        ROUND(MAX(amount), 2) AS max_amount,
        ROUND(AVG(distance_from_home), 2) AS avg_distance_from_home,
        ROUND(AVG(num_transactions_last_hour), 2) AS avg_num_transactions_last_hour,
        ROUND(AVG(total_amount_last_hour), 2) AS avg_total_amount_last_hour,
        ROUND(AVG(max_single_amount_last_hour), 2) AS avg_max_single_amount_last_hour
    FROM transactions_treated
    GROUP BY is_fraud
    ORDER BY is_fraud
""").show(truncate=False)

+--------+----------+----------+----------+----------------------+------------------------------+--------------------------+-------------------------------+
|is_fraud|avg_amount|min_amount|max_amount|avg_distance_from_home|avg_num_transactions_last_hour|avg_total_amount_last_hour|avg_max_single_amount_last_hour|
+--------+----------+----------+----------+----------------------+------------------------------+--------------------------+-------------------------------+
|false   |30242.54  |17.95     |1240629.47|0.17                  |408.26                        |1.985783734E7             |1719986.75                     |
|true    |118773.59 |0.01      |6253152.62|0.92                  |412.67                        |2.015499552E7             |1752255.49                     |
+--------+----------+----------+----------+----------------------+------------------------------+--------------------------+-------------------------------+



## Taxa de fraude por merchant_category

In [18]:
spark.sql("""
    SELECT
        merchant_category,
        COUNT(*) AS total_transactions,
        SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) AS fraud_transactions,
        ROUND(100.0 * SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) / COUNT(*), 2) AS fraud_rate_percentage
    FROM transactions_treated
    GROUP BY merchant_category
    ORDER BY fraud_rate_percentage DESC, total_transactions DESC
    LIMIT 15
""").show(truncate=False)

+-----------------+------------------+------------------+---------------------+
|merchant_category|total_transactions|fraud_transactions|fraud_rate_percentage|
+-----------------+------------------+------------------+---------------------+
|Travel           |935790            |187477            |20.03                |
|Grocery          |934029            |186987            |20.02                |
|Restaurant       |936178            |186951            |19.97                |
|Gas              |935401            |186829            |19.97                |
|Entertainment    |936173            |186890            |19.96                |
|Education        |933542            |186203            |19.95                |
|Healthcare       |936770            |186769            |19.94                |
|Retail           |935883            |186613            |19.94                |
+-----------------+------------------+------------------+---------------------+



## Taxa de fraude por channel

In [19]:
spark.sql("""
    SELECT
        channel,
        COUNT(*) AS total_transactions,
        SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) AS fraud_transactions,
        ROUND(100.0 * SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) / COUNT(*), 2) AS fraud_rate_percentage,
        ROUND(AVG(amount), 2) AS avg_amount
    FROM transactions_treated
    GROUP BY channel
    ORDER BY fraud_rate_percentage DESC, total_transactions DESC
""").show(truncate=False)

+-------+------------------+------------------+---------------------+----------+
|channel|total_transactions|fraud_transactions|fraud_rate_percentage|avg_amount|
+-------+------------------+------------------+---------------------+----------+
|pos    |651047            |651047            |100.00               |118930.25 |
|mobile |2269578           |281150            |12.39                |41842.23  |
|web    |4563141           |562522            |12.33                |40819.2   |
+-------+------------------+------------------+---------------------+----------+



## Taxa de fraude por pais

In [20]:
spark.sql("""
    SELECT
        country,
        COUNT(*) AS total_transactions,
        SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) AS fraud_transactions,
        ROUND(100.0 * SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) / COUNT(*), 2) AS fraud_rate_percentage,
        ROUND(AVG(amount), 2) AS avg_amount
    FROM transactions_treated
    GROUP BY country
    ORDER BY fraud_rate_percentage DESC, total_transactions DESC
    LIMIT 15
""").show(truncate=False)

+---------+------------------+------------------+---------------------+----------+
|country  |total_transactions|fraud_transactions|fraud_rate_percentage|avg_amount|
+---------+------------------+------------------+---------------------+----------+
|Mexico   |785704            |298841            |38.03                |15179.9   |
|Russia   |793730            |299425            |37.72                |56661.24  |
|Brazil   |804800            |298629            |37.11                |3784.36   |
|Nigeria  |849840            |298600            |35.14                |308284.4  |
|Australia|496695            |37652             |7.58                 |791.69    |
|USA      |500060            |37312             |7.46                 |565.54    |
|Japan    |527393            |37592             |7.13                 |65310.79  |
|Germany  |524464            |37205             |7.09                 |502.4     |
|Canada   |532632            |37278             |7.00                 |696.53    |
|UK 

## Comportamento por horario da transacao

In [21]:
spark.sql("""
    SELECT
        transaction_hour,
        COUNT(*) AS total_transactions,
        SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) AS fraud_transactions,
        ROUND(100.0 * SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) / COUNT(*), 2) AS fraud_rate_percentage,
        ROUND(AVG(amount), 2) AS avg_amount
    FROM transactions_treated
    GROUP BY transaction_hour
    ORDER BY transaction_hour
""").show(24, truncate=False)

+----------------+------------------+------------------+---------------------+----------+
|transaction_hour|total_transactions|fraud_transactions|fraud_rate_percentage|avg_amount|
+----------------+------------------+------------------+---------------------+----------+
|0               |155759            |41519             |26.66                |54766.09  |
|1               |280136            |165999            |59.26                |82479.77  |
|2               |280472            |166025            |59.19                |82206.57  |
|3               |280031            |165621            |59.14                |82835.45  |
|4               |281466            |166418            |59.13                |82583.89  |
|5               |208529            |41829             |20.06                |47522.96  |
|6               |196875            |41622             |21.14                |48576.98  |
|7               |301176            |41318             |13.72                |42111.48  |
|8        

## Impacto das variaveis binarias

In [22]:
spark.sql("""
    SELECT
        high_risk_merchant,
        COUNT(*) AS total_transactions,
        SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) AS fraud_transactions,
        ROUND(100.0 * SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) / COUNT(*), 2) AS fraud_rate_percentage
    FROM transactions_treated
    GROUP BY high_risk_merchant
    ORDER BY high_risk_merchant
""").show(truncate=False)

spark.sql("""
    SELECT
        weekend_transaction,
        COUNT(*) AS total_transactions,
        SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) AS fraud_transactions,
        ROUND(100.0 * SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) / COUNT(*), 2) AS fraud_rate_percentage
    FROM transactions_treated
    GROUP BY weekend_transaction
    ORDER BY weekend_transaction
""").show(truncate=False)

spark.sql("""
    SELECT
        card_present,
        COUNT(*) AS total_transactions,
        SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) AS fraud_transactions,
        ROUND(100.0 * SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) / COUNT(*), 2) AS fraud_rate_percentage
    FROM transactions_treated
    GROUP BY card_present
    ORDER BY card_present
""").show(truncate=False)

+------------------+------------------+------------------+---------------------+
|high_risk_merchant|total_transactions|fraud_transactions|fraud_rate_percentage|
+------------------+------------------+------------------+---------------------+
|false             |5611803           |1120352           |19.96                |
|true              |1871963           |374367            |20.00                |
+------------------+------------------+------------------+---------------------+

+-------------------+------------------+------------------+---------------------+
|weekend_transaction|total_transactions|fraud_transactions|fraud_rate_percentage|
+-------------------+------------------+------------------+---------------------+
|false              |5554103           |1109277           |19.97                |
|true               |1929663           |385442            |19.97                |
+-------------------+------------------+------------------+---------------------+

+------------+------

# Etapa 4 - Divisao da base e balanceamento do treino

## Criacao da chave aleatoria para o split

In [23]:
split_dataframe = spark.sql("""
    SELECT
        *,
        rand(42) AS split_key
    FROM transactions_treated
""")

split_dataframe.createOrReplaceTempView("transactions_with_split")

## Separacao em treino, validacao e teste

In [24]:
train_dataframe = spark.sql("""
    SELECT
        merchant_category,
        merchant_type,
        amount,
        currency,
        country,
        city,
        city_size,
        card_type,
        card_present,
        device,
        channel,
        distance_from_home,
        high_risk_merchant,
        transaction_hour,
        weekend_transaction,
        num_transactions_last_hour,
        total_amount_last_hour,
        unique_merchants_last_hour,
        unique_countries_last_hour,
        max_single_amount_last_hour,
        is_fraud
    FROM transactions_with_split
    WHERE split_key < 0.70
""")

validation_dataframe = spark.sql("""
    SELECT
        merchant_category,
        merchant_type,
        amount,
        currency,
        country,
        city,
        city_size,
        card_type,
        card_present,
        device,
        channel,
        distance_from_home,
        high_risk_merchant,
        transaction_hour,
        weekend_transaction,
        num_transactions_last_hour,
        total_amount_last_hour,
        unique_merchants_last_hour,
        unique_countries_last_hour,
        max_single_amount_last_hour,
        is_fraud
    FROM transactions_with_split
    WHERE split_key >= 0.70 AND split_key < 0.85
""")

test_dataframe = spark.sql("""
    SELECT
        merchant_category,
        merchant_type,
        amount,
        currency,
        country,
        city,
        city_size,
        card_type,
        card_present,
        device,
        channel,
        distance_from_home,
        high_risk_merchant,
        transaction_hour,
        weekend_transaction,
        num_transactions_last_hour,
        total_amount_last_hour,
        unique_merchants_last_hour,
        unique_countries_last_hour,
        max_single_amount_last_hour,
        is_fraud
    FROM transactions_with_split
    WHERE split_key >= 0.85
""")

train_dataframe.createOrReplaceTempView("transactions_train")
validation_dataframe.createOrReplaceTempView("transactions_validation")
test_dataframe.createOrReplaceTempView("transactions_test")

## Tamanho dos conjuntos

In [25]:
spark.sql("""
    SELECT
        dataset,
        total_transactions,
        ROUND(100.0 * total_transactions / SUM(total_transactions) OVER (), 2) AS percentage
    FROM (
        SELECT 'train' AS dataset, COUNT(*) AS total_transactions FROM transactions_train
        UNION ALL
        SELECT 'validation' AS dataset, COUNT(*) AS total_transactions FROM transactions_validation
        UNION ALL
        SELECT 'test' AS dataset, COUNT(*) AS total_transactions FROM transactions_test
    ) t
    ORDER BY dataset
""").show(truncate=False)

+----------+------------------+----------+
|dataset   |total_transactions|percentage|
+----------+------------------+----------+
|test      |1123547           |15.01     |
|train     |5238627           |70.00     |
|validation|1121592           |14.99     |
+----------+------------------+----------+



## Distribuicao da classe antes do balanceamento

In [26]:
spark.sql("""
    SELECT *
    FROM (
        SELECT 'train' AS dataset, is_fraud, COUNT(*) AS total_transactions
        FROM transactions_train
        GROUP BY is_fraud

        UNION ALL

        SELECT 'validation' AS dataset, is_fraud, COUNT(*) AS total_transactions
        FROM transactions_validation
        GROUP BY is_fraud

        UNION ALL

        SELECT 'test' AS dataset, is_fraud, COUNT(*) AS total_transactions
        FROM transactions_test
        GROUP BY is_fraud
    ) t
    ORDER BY dataset, is_fraud
""").show(truncate=False)

+----------+--------+------------------+
|dataset   |is_fraud|total_transactions|
+----------+--------+------------------+
|test      |false   |899294            |
|test      |true    |224253            |
|train     |false   |4191566           |
|train     |true    |1047061           |
|validation|false   |898187            |
|validation|true    |223405            |
+----------+--------+------------------+



## Balanceamento do conjunto de treino

In [27]:
balanced_train_dataframe = spark.sql("""
    SELECT
        merchant_category,
        merchant_type,
        amount,
        currency,
        country,
        city,
        city_size,
        card_type,
        card_present,
        device,
        channel,
        distance_from_home,
        high_risk_merchant,
        transaction_hour,
        weekend_transaction,
        num_transactions_last_hour,
        total_amount_last_hour,
        unique_merchants_last_hour,
        unique_countries_last_hour,
        max_single_amount_last_hour,
        is_fraud
    FROM transactions_train
    WHERE is_fraud = true

    UNION ALL

    SELECT
        merchant_category,
        merchant_type,
        amount,
        currency,
        country,
        city,
        city_size,
        card_type,
        card_present,
        device,
        channel,
        distance_from_home,
        high_risk_merchant,
        transaction_hour,
        weekend_transaction,
        num_transactions_last_hour,
        total_amount_last_hour,
        unique_merchants_last_hour,
        unique_countries_last_hour,
        max_single_amount_last_hour,
        is_fraud
    FROM (
        SELECT
            merchant_category,
            merchant_type,
            amount,
            currency,
            country,
            city,
            city_size,
            card_type,
            card_present,
            device,
            channel,
            distance_from_home,
            high_risk_merchant,
            transaction_hour,
            weekend_transaction,
            num_transactions_last_hour,
            total_amount_last_hour,
            unique_merchants_last_hour,
            unique_countries_last_hour,
            max_single_amount_last_hour,
            is_fraud,
            ROW_NUMBER() OVER (ORDER BY rand(42)) AS row_num
        FROM transactions_train
        WHERE is_fraud = false
    ) non_fraud_sample
    WHERE row_num <= (
        SELECT COUNT(*)
        FROM transactions_train
        WHERE is_fraud = true
    )
""")

balanced_train_dataframe.createOrReplaceTempView("transactions_train_balanced")

## Distribuicao da classe depois do balanceamento

In [28]:
spark.sql("""
    SELECT
        is_fraud,
        COUNT(*) AS total_transactions,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentage
    FROM transactions_train_balanced
    GROUP BY is_fraud
    ORDER BY is_fraud
""").show(truncate=False)

+--------+------------------+----------+
|is_fraud|total_transactions|percentage|
+--------+------------------+----------+
|false   |1047061           |50.00     |
|true    |1047061           |50.00     |
+--------+------------------+----------+



## Conferencia final dos conjuntos para a modelagem

In [29]:
spark.sql("""
    SELECT 'train_balanced' AS dataset, COUNT(*) AS total_transactions FROM transactions_train_balanced
    UNION ALL
    SELECT 'validation' AS dataset, COUNT(*) AS total_transactions FROM transactions_validation
    UNION ALL
    SELECT 'test' AS dataset, COUNT(*) AS total_transactions FROM transactions_test
""").show(truncate=False)

+--------------+------------------+
|dataset       |total_transactions|
+--------------+------------------+
|train_balanced|2094122           |
|validation    |1121592           |
|test          |1123547           |
+--------------+------------------+



# Etapa 5 - Preparacao para modelagem, treinamento e avaliacao

## Observacao sobre o treino dos modelos

A preparacao dos dados foi feita com Spark SQL. O treinamento dos algoritmos e realizado com PySpark MLlib, porque o Spark SQL nao oferece funcoes nativas para treinar Decision Tree, Logistic Regression e Redes Neurais.

## Criacao dos conjuntos finais para a modelagem

In [30]:
model_train_dataframe = spark.sql("""
    SELECT
        merchant_category,
        country,
        city_size,
        card_type,
        device,
        channel,
        amount,
        distance_from_home,
        CASE WHEN high_risk_merchant = true THEN 1 ELSE 0 END AS high_risk_merchant_num,
        transaction_hour,
        CASE WHEN weekend_transaction = true THEN 1 ELSE 0 END AS weekend_transaction_num,
        num_transactions_last_hour,
        total_amount_last_hour,
        unique_merchants_last_hour,
        unique_countries_last_hour,
        max_single_amount_last_hour,
        CASE WHEN is_fraud = true THEN 1.0 ELSE 0.0 END AS label
    FROM transactions_train_balanced
""")

model_validation_dataframe = spark.sql("""
    SELECT
        merchant_category,
        country,
        city_size,
        card_type,
        device,
        channel,
        amount,
        distance_from_home,
        CASE WHEN high_risk_merchant = true THEN 1 ELSE 0 END AS high_risk_merchant_num,
        transaction_hour,
        CASE WHEN weekend_transaction = true THEN 1 ELSE 0 END AS weekend_transaction_num,
        num_transactions_last_hour,
        total_amount_last_hour,
        unique_merchants_last_hour,
        unique_countries_last_hour,
        max_single_amount_last_hour,
        CASE WHEN is_fraud = true THEN 1.0 ELSE 0.0 END AS label
    FROM transactions_validation
""")

model_test_dataframe = spark.sql("""
    SELECT
        merchant_category,
        country,
        city_size,
        card_type,
        device,
        channel,
        amount,
        distance_from_home,
        CASE WHEN high_risk_merchant = true THEN 1 ELSE 0 END AS high_risk_merchant_num,
        transaction_hour,
        CASE WHEN weekend_transaction = true THEN 1 ELSE 0 END AS weekend_transaction_num,
        num_transactions_last_hour,
        total_amount_last_hour,
        unique_merchants_last_hour,
        unique_countries_last_hour,
        max_single_amount_last_hour,
        CASE WHEN is_fraud = true THEN 1.0 ELSE 0.0 END AS label
    FROM transactions_test
""")

## Pipeline de preparacao das features

In [31]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import DecisionTreeClassifier, LogisticRegression, MultilayerPerceptronClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler

categorical_columns = [
    "merchant_category",
    "country",
    "city_size",
    "card_type",
    "device",
    "channel"
]

numeric_columns = [
    "amount",
    "distance_from_home",
    "high_risk_merchant_num",
    "transaction_hour",
    "weekend_transaction_num",
    "num_transactions_last_hour",
    "total_amount_last_hour",
    "unique_merchants_last_hour",
    "unique_countries_last_hour",
    "max_single_amount_last_hour"
]

indexers = [
    StringIndexer(inputCol=column_name, outputCol=f"{column_name}_index", handleInvalid="keep")
    for column_name in categorical_columns
]

encoder = OneHotEncoder(
    inputCols=[f"{column_name}_index" for column_name in categorical_columns],
    outputCols=[f"{column_name}_ohe" for column_name in categorical_columns],
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=numeric_columns + [f"{column_name}_ohe" for column_name in categorical_columns],
    outputCol="features"
)

preprocess_pipeline = Pipeline(stages=indexers + [encoder, assembler])
preprocess_model = preprocess_pipeline.fit(model_train_dataframe)

train_prepared_dataframe = preprocess_model.transform(model_train_dataframe).select("features", "label")
validation_prepared_dataframe = preprocess_model.transform(model_validation_dataframe).select("features", "label")
test_prepared_dataframe = preprocess_model.transform(model_test_dataframe).select("features", "label")

In [32]:
feature_dimension = len(train_prepared_dataframe.select("features").first()[0])
print(f"Numero de features geradas pelo pipeline: {feature_dimension}")

Numero de features geradas pelo pipeline: 55


## Funcao auxiliar para avaliacao

In [33]:
multiclass_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
binary_evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

def evaluate_predictions(predictions_dataframe, model_name, dataset_name):
    return {
        "model": model_name,
        "dataset": dataset_name,
        "accuracy": round(multiclass_evaluator.setMetricName("accuracy").evaluate(predictions_dataframe), 4),
        "f1": round(multiclass_evaluator.setMetricName("f1").evaluate(predictions_dataframe), 4),
        "weighted_precision": round(multiclass_evaluator.setMetricName("weightedPrecision").evaluate(predictions_dataframe), 4),
        "weighted_recall": round(multiclass_evaluator.setMetricName("weightedRecall").evaluate(predictions_dataframe), 4),
        "auc_roc": round(binary_evaluator.evaluate(predictions_dataframe), 4)
    }

## Decision Tree

In [34]:
decision_tree_classifier = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=8,
    seed=42
)

decision_tree_model = decision_tree_classifier.fit(train_prepared_dataframe)
decision_tree_validation_predictions = decision_tree_model.transform(validation_prepared_dataframe)
decision_tree_test_predictions = decision_tree_model.transform(test_prepared_dataframe)

## Logistic Regression

In [35]:
logistic_regression_classifier = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    maxIter=30,
    regParam=0.0,
    elasticNetParam=0.0
)

logistic_regression_model = logistic_regression_classifier.fit(train_prepared_dataframe)
logistic_regression_validation_predictions = logistic_regression_model.transform(validation_prepared_dataframe)
logistic_regression_test_predictions = logistic_regression_model.transform(test_prepared_dataframe)

## Rede Neural

In [36]:
mlp_layers = [feature_dimension, 16, 8, 2]

multilayer_perceptron_classifier = MultilayerPerceptronClassifier(
    labelCol="label",
    featuresCol="features",
    layers=mlp_layers,
    maxIter=25,
    seed=42,
    blockSize=128
)

multilayer_perceptron_model = multilayer_perceptron_classifier.fit(train_prepared_dataframe)
multilayer_perceptron_validation_predictions = multilayer_perceptron_model.transform(validation_prepared_dataframe)
multilayer_perceptron_test_predictions = multilayer_perceptron_model.transform(test_prepared_dataframe)

## Comparacao das metricas

In [37]:
evaluation_results = [
    evaluate_predictions(decision_tree_validation_predictions, "Decision Tree", "validation"),
    evaluate_predictions(decision_tree_test_predictions, "Decision Tree", "test"),
    evaluate_predictions(logistic_regression_validation_predictions, "Logistic Regression", "validation"),
    evaluate_predictions(logistic_regression_test_predictions, "Logistic Regression", "test"),
    evaluate_predictions(multilayer_perceptron_validation_predictions, "Rede Neural", "validation"),
    evaluate_predictions(multilayer_perceptron_test_predictions, "Rede Neural", "test")
]

evaluation_dataframe = spark.createDataFrame(evaluation_results)
evaluation_dataframe.createOrReplaceTempView("model_evaluation_results")

In [38]:
spark.sql("""
    SELECT
        model,
        dataset,
        accuracy,
        f1,
        weighted_precision,
        weighted_recall,
        auc_roc
    FROM model_evaluation_results
    ORDER BY dataset, f1 DESC, auc_roc DESC
""").show(truncate=False)

+-------------------+----------+--------+------+------------------+---------------+-------+
|model              |dataset   |accuracy|f1    |weighted_precision|weighted_recall|auc_roc|
+-------------------+----------+--------+------+------------------+---------------+-------+
|Decision Tree      |test      |0.9278  |0.9306|0.939             |0.9278         |0.8859 |
|Logistic Regression|test      |0.9075  |0.9117|0.9233            |0.9075         |0.9683 |
|Rede Neural        |test      |0.2252  |0.1301|0.6856            |0.2252         |0.5241 |
|Decision Tree      |validation|0.9274  |0.9303|0.9388            |0.9274         |0.8862 |
|Logistic Regression|validation|0.9073  |0.9114|0.9231            |0.9073         |0.9679 |
|Rede Neural        |validation|0.2249  |0.1299|0.6869            |0.2249         |0.5236 |
+-------------------+----------+--------+------+------------------+---------------+-------+



## Matrizes de confusao no conjunto de teste

In [39]:
decision_tree_test_predictions.selectExpr("CAST(label AS INT) AS label", "CAST(prediction AS INT) AS prediction").createOrReplaceTempView("decision_tree_test_predictions")
logistic_regression_test_predictions.selectExpr("CAST(label AS INT) AS label", "CAST(prediction AS INT) AS prediction").createOrReplaceTempView("logistic_regression_test_predictions")
multilayer_perceptron_test_predictions.selectExpr("CAST(label AS INT) AS label", "CAST(prediction AS INT) AS prediction").createOrReplaceTempView("multilayer_perceptron_test_predictions")

In [40]:
spark.sql("SELECT 'Decision Tree' AS model, label, prediction, COUNT(*) AS total FROM decision_tree_test_predictions GROUP BY label, prediction ORDER BY label, prediction").show(truncate=False)

spark.sql("SELECT 'Logistic Regression' AS model, label, prediction, COUNT(*) AS total FROM logistic_regression_test_predictions GROUP BY label, prediction ORDER BY label, prediction").show(truncate=False)

spark.sql("SELECT 'Rede Neural' AS model, label, prediction, COUNT(*) AS total FROM multilayer_perceptron_test_predictions GROUP BY label, prediction ORDER BY label, prediction").show(truncate=False)

+-------------+-----+----------+------+
|model        |label|prediction|total |
+-------------+-----+----------+------+
|Decision Tree|0    |0         |831150|
|Decision Tree|0    |1         |68144 |
|Decision Tree|1    |0         |13027 |
|Decision Tree|1    |1         |211226|
+-------------+-----+----------+------+

+-------------------+-----+----------+------+
|model              |label|prediction|total |
+-------------------+-----+----------+------+
|Logistic Regression|0    |0         |814516|
|Logistic Regression|0    |1         |84778 |
|Logistic Regression|1    |0         |19121 |
|Logistic Regression|1    |1         |205132|
+-------------------+-----+----------+------+

+-----------+-----+----------+------+
|model      |label|prediction|total |
+-----------+-----+----------+------+
|Rede Neural|0    |0         |37857 |
|Rede Neural|0    |1         |861437|
|Rede Neural|1    |0         |9068  |
|Rede Neural|1    |1         |215185|
+-----------+-----+----------+------+



## Melhor modelo no conjunto de teste

In [41]:
spark.sql("""
    SELECT
        model,
        accuracy,
        f1,
        weighted_precision,
        weighted_recall,
        auc_roc
    FROM model_evaluation_results
    WHERE dataset = 'test'
    ORDER BY f1 DESC, auc_roc DESC
""").show(truncate=False)

+-------------------+--------+------+------------------+---------------+-------+
|model              |accuracy|f1    |weighted_precision|weighted_recall|auc_roc|
+-------------------+--------+------+------------------+---------------+-------+
|Decision Tree      |0.9278  |0.9306|0.939             |0.9278         |0.8859 |
|Logistic Regression|0.9075  |0.9117|0.9233            |0.9075         |0.9683 |
|Rede Neural        |0.2252  |0.1301|0.6856            |0.2252         |0.5241 |
+-------------------+--------+------+------------------+---------------+-------+

